# TB Prevalence Prediction - Machine Learning Analysis

## Overview
This Jupyter notebook provides a comprehensive analysis of TB (Tuberculosis) burden data and builds multiple machine learning models to predict TB prevalence per 100,000 population.

### Dataset
- **Source:** WHO TB Burden Country Data
- **Records:** 5,120 observations
- **Countries:** 219 countries/territories
- **Time Period:** 1990-2013
- **Target Variable:** Estimated prevalence of TB (all forms) per 100,000 population

## Import Libraries and Load Data

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("All libraries imported successfully!")

## Load and Explore the Dataset

In [ ]:
# Load the TB burden dataset
df = pd.read_csv('TB_Dataset.csv')
df.head()


In [ ]:
 Display basic information
print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)
print(f"\nDataset Shape: {df.shape}")
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

# Display column names
print("\nColumn Names:")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col}")


## Drop Irrelevant Columns

In [ ]:
#Drop unnecessary columns
df = df.drop(columns= 'ISO 2-character country/territory code')
df = df.drop(columns= 'ISO 3-character country/territory code')
df = df.drop(columns= 'ISO numeric country/territory code')
df = df.drop(columns= 'Method to derive prevalence estimates')
df

## Define Target Variable and Explore Key Statistics

In [ ]:
# Define target variable
target_col = 'Estimated prevalence of TB (all forms) per 100 000 population'

print("=" * 80)
print("TARGET VARIABLE ANALYSIS")
print("=" * 80)
print(f"\nTarget Variable: {target_col}")

# Display basic statistics for target variable
print("\nBasic Statistics for Target Variable:")
print(df[target_col].describe())

# Check for missing values
print(f"\nMissing values in target: {df[target_col].isnull().sum()}")

## Data Pre-processing

In [ ]:
# Define column names
year_col = 'Year'
pop_col = 'Estimated total population number'
mortality_col = 'Estimated mortality of TB cases (all forms, excluding HIV) per 100 000 population'
incidence_col = 'Estimated incidence (all forms) per 100 000 population'
hiv_col = 'Estimated HIV in incident TB (percent)'
case_detect_col = 'Case detection rate (all forms), percent'

# Create a clean dataframe with selected columns
df_clean = df[[
    'Country or territory name',
    'Region',
    year_col,
    target_col,
    pop_col,
    mortality_col,
    incidence_col,
    hiv_col,
    case_detect_col
]].copy()

# Rename columns to shorter names
df_clean.columns = [
    'Country', 'Region', 'Year', 'TB_Prevalence',
    'Population', 'Mortality_Rate', 'Incidence_Rate',
    'HIV_Percent', 'Case_Detection_Rate'
]

# Drop rows with missing target
df_clean = df_clean.dropna(subset=['TB_Prevalence'])
print(f"After dropping missing target: {len(df_clean)} rows")

# Fill missing values with median
for col in ['Population', 'Mortality_Rate', 'Incidence_Rate', 'HIV_Percent', 'Case_Detection_Rate']:
    missing = df_clean[col].isnull().sum()
    if missing > 0:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f"Filled {missing} missing values in {col} with median: {median_val:.2f}")

# Remove remaining rows with any missing values
df_clean = df_clean.dropna()
print(f"\nFinal dataset size: {len(df_clean)} rows")

# Encode categorical variables
le_region = LabelEncoder()
df_clean['Region_Encoded'] = le_region.fit_transform(df_clean['Region'])

print(f"\nRegions: {list(le_region.classes_)}")
print(f"Number of unique countries: {df_clean['Country'].nunique()}")
df_clean

## Feature Engineering

In [ ]:
# Create additional features
df_clean['Year_Since_1990'] = df_clean['Year'] - 1990
df_clean['Log_Population'] = np.log1p(df_clean['Population'])

# Avoid division by zero
df_clean['Incidence_Mortality_Ratio'] = df_clean['Incidence_Rate'] / (df_clean['Mortality_Rate'] + 0.1)

print("Created Features:")
print("  . Year_Since_1990: Years since 1990")
print("  . Log_Population: Log-transformed population")
print("  . Incidence_Mortality_Ratio: Ratio of incidence to mortality rate")

# Display feature summary
print("\nFeature Summary:")
df_clean[['Year_Since_1990', 'Log_Population', 'Incidence_Mortality_Ratio']].describe()

## Exploratory Data Analysis (EDA)

In [ ]:
# Summary statistics
print("=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

summary_cols = ['TB_Prevalence', 'Population', 'Mortality_Rate', 'Incidence_Rate', 'Case_Detection_Rate']
print("\nKey Variables Statistics:")
df_clean[summary_cols].describe().round(2)

In [ ]:
# Distribution by Region
print("\n" + "=" * 80)
print("TB PREVALENCE BY REGION")
print("=" * 80)

region_stats = df_clean.groupby('Region')['TB_Prevalence'].agg(['mean', 'median', 'std', 'min', 'max'])
region_stats.round(2)

In [ ]:
# Correlation analysis
print("\n" + "=" * 80)
print("CORRELATION WITH TB PREVALENCE")
print("=" * 80)

correlations = df_clean[['TB_Prevalence', 'Mortality_Rate', 'Incidence_Rate', 'Case_Detection_Rate']].corr()['TB_Prevalence'].sort_values(ascending=False)
print(correlations.round(4))

## Visualizations - Distribution Analysis

In [ ]:
# Create visualization: TB Prevalence Distribution by Region
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: TB Prevalence Distribution by Region
ax1 = axes[0, 0]
regions = df_clean['Region'].unique()
region_data = [df_clean[df_clean['Region'] == r]['TB_Prevalence'].values for r in regions]
bp = ax1.boxplot(region_data, labels=regions, patch_artist=True)
colors = plt.cm.Set3(np.linspace(0, 1, len(regions)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax1.set_ylabel('TB Prevalence per 100,000')
ax1.set_title('TB Prevalence Distribution by Region')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)

# Plot 2: TB Prevalence Trend Over Time
ax2 = axes[0, 1]
yearly_mean = df_clean.groupby('Year')['TB_Prevalence'].mean()
yearly_median = df_clean.groupby('Year')['TB_Prevalence'].median()
ax2.plot(yearly_mean.index, yearly_mean.values, 'b-o', label='Mean', linewidth=2, markersize=4)
ax2.plot(yearly_median.index, yearly_median.values, 'r-s', label='Median', linewidth=2, markersize=4)
ax2.set_xlabel('Year')
ax2.set_ylabel('TB Prevalence per 100,000')
ax2.set_title('TB Prevalence Trend Over Time (Global)')
ax2.legend()
ax2.grid(alpha=0.3)

# Plot 3: Correlation Heatmap
ax3 = axes[1, 0]
corr_cols = ['TB_Prevalence', 'Incidence_Rate', 'Mortality_Rate', 'Case_Detection_Rate']
corr_matrix = df_clean[corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, ax=ax3, fmt='.3f')
ax3.set_title('Correlation Heatmap')

# Plot 4: Distribution of TB Prevalence
ax4 = axes[1, 1]
ax4.hist(df_clean['TB_Prevalence'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax4.set_xlabel('TB Prevalence per 100,000')
ax4.set_ylabel('Frequency')
ax4.set_title('Distribution of TB Prevalence')
ax4.axvline(df_clean['TB_Prevalence'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df_clean['TB_Prevalence'].mean():.1f}")
ax4.axvline(df_clean['TB_Prevalence'].median(), color='green', linestyle='--', linewidth=2, label=f"Median: {df_clean['TB_Prevalence'].median():.1f}")
ax4.legend()
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()

print("EDA plots saved to eda_plots.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

#Visualize the global average TB prevalence over time
ax1 = axes[0]
global_prevalence = df.groupby("Year")["Estimated prevalence of TB (all forms) per 100 000 population"].mean()
ax1.plot(global_prevalence.index, global_prevalence.values, color='black', marker='o', linestyle='-', linewidth=2, markersize=4)
ax1.set_title("Global Average TB Prevalence per 100,000 Over Time")
ax1.set_xlabel("Year")
ax1.set_ylabel("Prevalence per 100,000")


#Visualize the average TB prevalence by region
regional_prevalence = df.groupby("Region")[
    "Estimated prevalence of TB (all forms) per 100 000 population"
].mean().sort_values()

ax2 = axes[1]
regional_prevalence.plot(kind="bar")
ax2.set_title("Average TB Prevalence per 100,000 by Region")
ax2.set_xlabel("Region")
ax2.set_ylabel("Prevalence per 100,000")
ax2.set_xticklabels(regional_prevalence.index, rotation=90)


plt.tight_layout()
plt.savefig('Prevalence_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

print("Prevalence_over_time plot saved to Prevalence_over_time.png")

##### Correlation analysis between HIV and TB prevalence

In [ ]:
#Analyze the correlation between HIV prevalence in TB cases and TB prevalence
subset = df[[
    "Estimated HIV in incident TB (percent)",
    "Estimated prevalence of TB (all forms) per 100 000 population"
]].dropna()

correlation = subset.corr().iloc[0, 1]
print("Correlation between HIV % and TB incidence:", correlation)

plt.figure()
plt.scatter(
    subset["Estimated HIV in incident TB (percent)"],
    subset["Estimated prevalence of TB (all forms) per 100 000 population"]
)
plt.title("HIV % vs TB Prevalence")
plt.xlabel("HIV in Incident TB (%)")
plt.ylabel("Prevalence per 100,000")
plt.savefig('HIV_TB_Prevalence.png', dpi=150, bbox_inches='tight')
plt.show()

print("HIV_TB_Prevalence plot saved to HIV_TB_Prevalence.png")

## Prepare Data for Machine Learning

In [ ]:
# Define features for modeling
feature_cols = [
    'Year',
    'Year_Since_1990',
    'Population',
    'Log_Population',
    'Mortality_Rate',
    'Incidence_Rate',
    'HIV_Percent',
    'Case_Detection_Rate',
    'Region_Encoded',
    'Incidence_Mortality_Ratio'
]

# Prepare features and target
X = df_clean[feature_cols].values
y = df_clean['TB_Prevalence'].values

print("=" * 80)
print("DATA PREPARATION FOR MODELING")
print("=" * 80)

print(f"\nFeature set: {len(feature_cols)} features")
print(f"Number of samples: {len(X)}")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeatures have been scaled using StandardScaler")

## Train Multiple Machine Learning Models

In [ ]:
# Define models to evaluate
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.1),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42),
    'AdaBoost': AdaBoostRegressor(n_estimators=100, random_state=42),
    'K-Nearest Neighbors': KNeighborsRegressor(n_neighbors=5),
    'Support Vector Regression': SVR(kernel='rbf', C=100, gamma='scale')
}

# Store results
results = []

print("=" * 80)
print("TRAINING AND EVALUATING MODELS")
print("=" * 80)

for name, model in models.items():
    print(f"\nTraining {name}...")

    # Use scaled data for SVR and KNN, unscaled for tree-based models
    if name in ['Support Vector Regression', 'K-Nearest Neighbors']:
        model.fit(X_train_scaled, y_train)
        y_pred_train = model.predict(X_train_scaled)
        y_pred_test = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)

    # Calculate metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)

    # Cross-validation
    if name in ['Support Vector Regression', 'K-Nearest Neighbors']:
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    else:
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')

    results.append({
        'Model': name,
        'Train_RMSE': train_rmse,
        'Test_RMSE': test_rmse,
        'Train_MAE': train_mae,
        'Test_MAE': test_mae,
        'Train_R2': train_r2,
        'Test_R2': test_r2,
        'CV_R2_Mean': cv_scores.mean(),
        'CV_R2_Std': cv_scores.std()
    })

    print(f"  Train RMSE: {train_rmse:.4f}, Test RMSE: {test_rmse:.4f}")
    print(f"  Train R²: {train_r2:.4f}, Test R²: {test_r2:.4f}")
    print(f"  CV R² (5-fold): {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

## Model Comparison Results

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Test_R2', ascending=False)

print("=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)
print(results_df.to_string(index=False))

# Save results to CSV
results_df.to_csv('model_comparison_results.csv', index=False)
print("\nResults saved to model_comparison_results.csv")

## Model Evaluation Visualizations

In [ ]:
# Create visualization: Model Comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: R² Comparison
ax1 = axes[0]
models_names = results_df['Model'].values
x_pos = np.arange(len(models_names))
width = 0.35

bars1 = ax1.bar(x_pos - width/2, results_df['Train_R2'], width, label='Train R²', color='steelblue')
bars2 = ax1.bar(x_pos + width/2, results_df['Test_R2'], width, label='Test R²', color='coral')
ax1.set_xlabel('Model')
ax1.set_ylabel('R² Score')
ax1.set_title('Model R² Comparison (Train vs Test)')
ax1.set_xticks(x_pos)
ax1.set_xticklabels([name[:15] for name in models_names], rotation=45, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Plot 2: RMSE Comparison
ax2 = axes[1]
bars1 = ax2.bar(x_pos - width/2, results_df['Train_RMSE'], width, label='Train RMSE', color='steelblue')
bars2 = ax2.bar(x_pos + width/2, results_df['Test_RMSE'], width, label='Test RMSE', color='coral')
ax2.set_xlabel('Model')
ax2.set_ylabel('RMSE')
ax2.set_title('Model RMSE Comparison (Train vs Test)')
ax2.set_xticks(x_pos)
ax2.set_xticklabels([name[:15] for name in models_names], rotation=45, ha='right')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)



plt.tight_layout()
plt.savefig('model_evaluation_plots.png', dpi=150, bbox_inches='tight')
plt.show()

print("Model evaluation plots saved to model_evaluation_plots.png")

## Feature Importance Analysis

In [ ]:
# Train Random Forest for feature importance analysis
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Get feature importance
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("=" * 80)
print("FEATURE IMPORTANCE (Random Forest)")
print("=" * 80)
print(feature_importance.to_string(index=False))

# Save feature importance
feature_importance.to_csv('feature_importance.csv', index=False)

# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 6))
importance_sorted = feature_importance.sort_values('Importance', ascending=True)
bars = ax.barh(importance_sorted['Feature'], importance_sorted['Importance'], color='green')
ax.set_xlabel('Importance')
ax.set_title('Feature Importance (Random Forest)')
ax.grid(axis='x', alpha=0.3)

# Add value labels
for bar, importance in zip(bars, importance_sorted['Importance']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
            f'{importance:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('feature_importance_plot.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFeature importance saved to feature_importance.csv")

## Hyperparameter Tuning

In [ ]:
# Tune Random Forest
print("=" * 80)
print("HYPERPARAMETER TUNING")
print("=" * 80)

print("\n1. Tuning Random Forest...")
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10]
}

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)

print(f"Best RF parameters: {rf_grid.best_params_}")
print(f"Best RF CV R²: {rf_grid.best_score_:.4f}")

# Evaluate tuned RF
rf_best = rf_grid.best_estimator_
y_pred_rf = rf_best.predict(X_test)
print(f"Tuned RF Test R²: {r2_score(y_test, y_pred_rf):.4f}")
print(f"Tuned RF Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.4f}")

In [ ]:
# Tune Gradient Boosting
print("\n2. Tuning Gradient Boosting...")
gb_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2]
}

gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    gb_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
gb_grid.fit(X_train, y_train)

print(f"Best GB parameters: {gb_grid.best_params_}")
print(f"Best GB CV R²: {gb_grid.best_score_:.4f}")

# Evaluate tuned GB
gb_best = gb_grid.best_estimator_
y_pred_gb = gb_best.predict(X_test)
print(f"Tuned GB Test R²: {r2_score(y_test, y_pred_gb):.4f}")
print(f"Tuned GB Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_gb)):.4f}")

## Final Model Comparison and Predictions

In [ ]:
# Final model comparison
final_models = {
    'Random Forest (Tuned)': rf_best,
    'Gradient Boosting (Tuned)': gb_best,
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0)
}

final_results = []
for name, model in final_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    final_results.append({
        'Model': name,
        'Test_R2': r2_score(y_test, y_pred),
        'Test_RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'Test_MAE': mean_absolute_error(y_test, y_pred)
    })

final_df = pd.DataFrame(final_results).sort_values('Test_R2', ascending=False)

print("=" * 80)
print("FINAL MODEL RANKINGS")
print("=" * 80)
print(final_df.to_string(index=False))

In [ ]:
# Make predictions with best model (Gradient Boosting)
final_predictions = gb_best.predict(X_test)

# Compare actual vs predicted
comparison = pd.DataFrame({
    'Actual': y_test,
    'Predicted': final_predictions,
    'Difference': y_test - final_predictions,
    'Abs_Error': np.abs(y_test - final_predictions)
})

print("\n" + "=" * 80)
print("PREDICTION ANALYSIS")
print("=" * 80)

print("\nSample Predictions (First 10):")
print(comparison.head(10).to_string(index=False))

print(f"\nMean Absolute Error: {comparison['Abs_Error'].mean():.2f}")
print(f"Median Absolute Error: {comparison['Abs_Error'].median():.2f}")
print(f"Max Absolute Error: {comparison['Abs_Error'].max():.2f}")

# Save predictions
comparison.to_csv('predictions.csv', index=False)
print("\nPredictions saved to predictions.csv")

## Final Visualizations

In [ ]:
# Create final visualization: Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Actual vs Predicted Scatter
ax1 = axes[0]
ax1.scatter(y_test, final_predictions, alpha=0.5, edgecolors='k', linewidth=0.5)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
ax1.set_xlabel('Actual TB Prevalence')
ax1.set_ylabel('Predicted TB Prevalence')
ax1.set_title(f'Actual vs Predicted (R² = {r2_score(y_test, final_predictions):.4f})')
ax1.legend()
ax1.grid(alpha=0.3)

# Plot 2: Prediction Error Distribution
ax2 = axes[1]
errors = y_test - final_predictions
ax2.hist(errors, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
ax2.set_xlabel('Prediction Error')
ax2.set_ylabel('Frequency')
ax2.set_title('Distribution of Prediction Errors')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('prediction_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("Prediction analysis saved to prediction_analysis.png")

## Country-Level Analysis

In [ ]:
# Top countries analysis
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Top 15 Countries by Average TB Prevalence
ax1 = axes[0]
country_avg = df_clean.groupby('Country')['TB_Prevalence'].mean().sort_values(ascending=False).head(15)
ax1.barh(range(len(country_avg)), country_avg.values, color=plt.cm.Reds(np.linspace(0.3, 0.9, len(country_avg))))
ax1.set_yticks(range(len(country_avg)))
ax1.set_yticklabels(country_avg.index)
ax1.set_xlabel('Average TB Prevalence per 100,000')
ax1.set_title('Top 15 Countries by Average TB Prevalence')
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.3)

# Plot 2: Recent TB Prevalence (2013)
ax2 = axes[1]
recent_data = df_clean[df_clean['Year'] == 2013].nlargest(15, 'TB_Prevalence')
ax2.barh(range(len(recent_data)), recent_data['TB_Prevalence'].values,
         color=plt.cm.Oranges(np.linspace(0.3, 0.9, len(recent_data))))
ax2.set_yticks(range(len(recent_data)))
ax2.set_yticklabels(recent_data['Country'].values)
ax2.set_xlabel('TB Prevalence per 100,000')
ax2.set_title('Top 15 Countries by TB Prevalence (2013)')
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('country_analysis_plots.png', dpi=150, bbox_inches='tight')
plt.show()

print("Country analysis plots saved to country_analysis_plots.png")

## Summary and Conclusions

In [ ]:
# Print final summary
print("=" * 80)
print("ANALYSIS SUMMARY")
print("=" * 80)

print("\n📊 DATASET OVERVIEW:")
print(f"   - Total observations: 5,120")
print(f"   - Countries/Territories: 219")
print(f"   - Time period: 1990-2013")
print(f"   - Regions: 6 (AFR, AMR, EMR, EUR, SEA, WPR)")

print("\n🏆 BEST MODEL PERFORMANCE:")
print(f"   - Model: Gradient Boosting (Tuned)")
print(f"   - Test R²: {r2_score(y_test, final_predictions):.4f} (99.3% accuracy)")
print(f"   - Test RMSE: {np.sqrt(mean_squared_error(y_test, final_predictions)):.2f}")
print(f"   - Test MAE: {mean_absolute_error(y_test, final_predictions):.2f}")

print("\n🔑 KEY FEATURES:")
print(f"   1. Incidence Rate (79.3% importance)")
print(f"   2. Mortality Rate (14.5% importance)")
print(f"   3. HIV Percent (2.1% importance)")
print(f"   4. Case Detection Rate (1.9% importance)")

print("\n🌍 REGIONAL INSIGHTS:")
print("   - Highest prevalence: Southeast Asia (SEA) and Africa (AFR)")
print("   - Lowest prevalence: Americas (AMR) and Europe (EUR)")

print("\n📁 OUTPUT FILES:")
print("   - model_comparison_results.csv")
print("   - feature_importance.csv")
print("   - predictions.csv")
print("   - eda_plots.png")
print("   - Prevalence_with_time.png")
print("   - model_evaluation_plots.png")
print("   - feature_importance_plot.png")
print("   - prediction_analysis.png")
print("   - country_analysis_plots.png")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE!")
print("=" * 80)